# Kayak Recommandation de Destinations Françaises

> *Pipeline Data Engineering complet : API Météo · Scraping Booking.com · AWS S3 · AWS RDS*

**Périmètre :** 35 villes touristiques françaises
**Stack :** OpenWeatherMap · BeautifulSoup · AWS S3 · AWS RDS · Plotly

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import re
import warnings
import requests
import time
warnings.filterwarnings('ignore')

print("✅ Imports OK")


## 2. Partie 1 — Données Météo (OpenWeatherMap + Nominatim)

In [ ]:
# Chargement des données météo collectées via OpenWeatherMap
# (voir kayak_part1_weather.py pour le code de collecte)

df_weather = pd.read_csv('../data/weather_france_cities.csv')
df_daily   = pd.read_csv('../data/weather_daily_details.csv')

print(f"Météo chargée : {len(df_weather)} villes, {len(df_daily)} observations journalières")
print(f"Colonnes météo : {df_weather.columns.tolist()}")
df_weather.head()


In [ ]:
# Score météo composite sur 7 jours par ville
# (calculé dans la collecte : température, probabilité de pluie, humidité)

df_weather_sorted = df_weather.sort_values('weather_score', ascending=False).reset_index(drop=True)

print("Top 10 destinations par score météo :")
print(df_weather_sorted[['city_name','weather_score','temp','precip_prob']].head(10).to_string(index=False))

TOP5 = df_weather_sorted.head(5)['city_name'].tolist()
print(f"\nTop 5 destinations : {TOP5}")


In [ ]:
# Visualisation : évolution météo journalière pour le Top 5

top5_daily = df_daily[df_daily['city'].isin(TOP5)].copy()

fig_daily = px.line(
    top5_daily, x='date', y='weather_score', color='city',
    markers=True,
    title="Évolution du Score Météo sur 7 jours — Top 5 Destinations",
    labels={'weather_score': 'Score météo', 'date': 'Date', 'city': 'Ville'}
)
fig_daily.update_layout(height=450, hovermode='x unified')
fig_daily.show()


In [ ]:
# Carte des 35 destinations

fig_dest = px.scatter_mapbox(
    df_weather_sorted,
    lat='latitude', lon='longitude',
    size='weather_score',
    color='weather_score',
    color_continuous_scale='RdYlGn',
    hover_name='city_name',
    hover_data={'weather_score': ':.1f', 'temp': ':.1f', 'precip_prob': ':.0f',
                'latitude': False, 'longitude': False},
    zoom=4.5,
    center={'lat': 46.5, 'lon': 2.5},
    mapbox_style='carto-positron',
    title='Score Météo — 35 Destinations Françaises',
    size_max=28,
    labels={'weather_score': 'Score météo', 'temp': 'Temp (°C)', 'precip_prob': 'Pluie (%)'}
)

top5_df = df_weather_sorted[df_weather_sorted['city_name'].isin(TOP5)]
fig_dest.add_trace(go.Scattermapbox(
    lat=top5_df['latitude'], lon=top5_df['longitude'],
    mode='markers+text',
    marker=dict(size=20, color='gold', symbol='star'),
    text=[' ' + c for c in top5_df['city_name']],
    textposition='top right',
    name='Top 5',
    textfont=dict(size=10, color='black')
))

fig_dest.update_layout(height=600, margin=dict(t=50, b=0, l=0, r=0))
fig_dest.show()


## 3. Partie 2 — Scraping Hôtels Booking.com

In [ ]:
# Chargement des hôtels scrapés via BeautifulSoup
# (voir kayak_part2_scraping.py pour le code de scraping)

df_hotels_raw = pd.read_csv('../data/hotels.csv')

print(f"Hôtels bruts : {len(df_hotels_raw)}")
print(df_hotels_raw.head(3))


In [ ]:
# Nettoyage de la colonne rating (format texte → float)

def extract_rating(rating_str):
    """Extrait la note numérique depuis le texte Booking.com."""
    if pd.isna(rating_str):
        return None
    match = re.search(r'(\d+[,.]\d+)', str(rating_str))
    if match:
        return float(match.group(1).replace(',', '.'))
    return None

df_hotels_raw['score'] = df_hotels_raw['rating'].apply(extract_rating)

# Jointure avec les coordonnées des villes (lat/lon depuis weather)
df_coords = df_weather[['city_name', 'latitude', 'longitude']].copy()
df_coords['city'] = df_coords['city_name'].str.title()

df_hotels = df_hotels_raw.merge(df_coords, on='city', how='left')
df_hotels = df_hotels.dropna(subset=['hotel_name', 'score'])
df_hotels['hotel_id'] = range(1, len(df_hotels) + 1)

print(f"{len(df_hotels)} hôtels nettoyés avec scores")
print(df_hotels[['city', 'hotel_name', 'score']].head(10).to_string(index=False))


In [ ]:
# 1. Top 20 hôtels par note
top20_hotels = df_hotels.dropna(subset=['score', 'latitude', 'longitude']) \
                         .sort_values('score', ascending=False) \
                         .head(20)
print("Top 20 hôtels les mieux notés :")
print(top20_hotels[['hotel_name', 'city', 'score']].to_string(index=False))

# 2. Géocodage précis via Nominatim (nom + ville)
def geocode_hotel(hotel_name, city):
    """Géocode un hôtel précis via Nominatim. Retourne (lat, lon) ou (None, None)."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": f"{hotel_name}, {city}, France", "format": "json", "limit": 1}
    headers = {"User-Agent": "KayakProject/1.0 (elat.martial@yahoo.fr)"}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        data = r.json()
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
    except Exception:
        pass
    return None, None

precise_lats, precise_lons = [], []
for _, row in top20_hotels.iterrows():
    lat, lon = geocode_hotel(row['hotel_name'], row['city'])
    if lat is None:
        lat, lon = row['latitude'], row['longitude']
    precise_lats.append(lat)
    precise_lons.append(lon)
    time.sleep(1)

top20_hotels = top20_hotels.copy()
top20_hotels['latitude'] = precise_lats
top20_hotels['longitude'] = precise_lons
print(f"\n✅ Géocodage terminé pour {len(top20_hotels)} hôtels")

# 3. Jitter (~5km) sur les hôtels en repli (coordonnées de ville dupliquées)
# pour une meilleure lisibilité visuelle sur la carte nationale
np.random.seed(42)
is_fallback = top20_hotels.duplicated(subset=['latitude', 'longitude'], keep=False)
jitter_lat = np.random.uniform(-0.05, 0.05, size=len(top20_hotels))
jitter_lon = np.random.uniform(-0.05, 0.05, size=len(top20_hotels))
top20_hotels.loc[is_fallback, 'latitude']  += jitter_lat[is_fallback]
top20_hotels.loc[is_fallback, 'longitude'] += jitter_lon[is_fallback]
print(f"✅ Jitter appliqué à {is_fallback.sum()} hôtels")

print(top20_hotels[['hotel_name', 'city', 'latitude', 'longitude']].to_string(index=False))


In [ ]:
# Carte Top 20 Hôtels

fig_hotels = px.scatter_mapbox(
    top20_hotels,
    lat='latitude', lon='longitude',
    color='city',
    size='score',
    size_max=20,
    hover_name='hotel_name',
    hover_data={'score': True, 'city': True, 'latitude': False, 'longitude': False},
    zoom=4.5,
    center={'lat': 46.5, 'lon': 2.5},
    mapbox_style='carto-positron',
    title='Top 20 Hôtels les Mieux Notés — Meilleures Destinations',
    labels={'score': 'Note Booking', 'city': 'Ville'}
)
fig_hotels.update_layout(height=600, margin=dict(t=50, b=0, l=0, r=0))
fig_hotels.show()


## 4. Partie 3 — ETL : AWS S3 → AWS RDS

In [ ]:
# Dataset final consolidé (météo + hôtels) pour l'entrepôt de données
df_final = df_weather_sorted.merge(
    df_hotels[['city', 'hotel_name', 'score']],
    left_on='city_name', right_on='city', how='left'
)
print(f"Dataset final : {df_final.shape}")
df_final.head()


In [ ]:
# ── Chargement sur AWS S3 (Data Lake) ──────────────────────────────────
# (voir kayak_part3_etl.py pour le code complet)
#
# Structure S3 :
#   s3://your-bucket/raw/weather_france_cities.csv
#   s3://your-bucket/raw/hotels_france.csv
#   s3://your-bucket/processed/final_kayak_data.csv

import boto3, io
from dotenv import load_dotenv
import os

load_dotenv()

try:
    s3 = boto3.client(
        's3',
        aws_access_key_id     = os.getenv('AWS_ACCESS_KEY_ID'),
        aws_secret_access_key = os.getenv('AWS_SECRET_ACCESS_KEY'),
        region_name           = os.getenv('AWS_DEFAULT_REGION', 'eu-west-3')
    )
    BUCKET = os.getenv('S3_BUCKET_NAME')

    def upload_to_s3(df, key):
        buf = io.StringIO()
        df.to_csv(buf, index=False)
        s3.put_object(Bucket=BUCKET, Key=key, Body=buf.getvalue())
        print(f"  ✅ s3://{BUCKET}/{key} ({len(df)} lignes)")

    print("📤 Upload vers S3...")
    upload_to_s3(df_weather,  "raw/weather_france_cities.csv")
    upload_to_s3(df_hotels,   "raw/hotels_france.csv")
    upload_to_s3(df_daily,    "raw/weather_daily_details.csv")
    upload_to_s3(df_final,    "processed/final_kayak_data.csv")
    print("\n✅ Data Lake S3 alimenté !")

except Exception as e:
    print(f"⚠️ AWS non configuré : {e}")
    print("→ Configurer les variables dans .env pour activer S3")


In [ ]:
# ── ETL S3 → RDS PostgreSQL (Data Warehouse) ───────────────────────────────
from sqlalchemy import create_engine

RDS_HOST = os.getenv('RDS_HOST')
RDS_PORT = os.getenv('RDS_PORT', '5432')
RDS_DB   = os.getenv('RDS_DB')
RDS_USER = os.getenv('RDS_USER')
RDS_PASS = os.getenv('RDS_PASSWORD')

try:
    engine = create_engine(f"postgresql+psycopg2://{RDS_USER}:{RDS_PASS}@{RDS_HOST}:{RDS_PORT}/{RDS_DB}")

    df_weather.to_sql('cities_weather', engine, if_exists='replace', index=False)
    print(f"  ✅ Table 'cities_weather' : {len(df_weather)} lignes")

    df_hotels.to_sql('hotels', engine, if_exists='replace', index=False)
    print(f"  ✅ Table 'hotels' : {len(df_hotels)} lignes")

    df_daily.to_sql('weather_daily', engine, if_exists='replace', index=False)
    print(f"  ✅ Table 'weather_daily' : {len(df_daily)} lignes")

    q = pd.read_sql("SELECT city_name, weather_score, temp FROM cities_weather ORDER BY weather_score DESC LIMIT 5", engine)
    print("\n📊 Top 5 depuis RDS :")
    print(q)

except Exception as e:
    print(f"⚠️ RDS non configuré : {e}")
    print("→ Configurer les variables RDS dans .env pour activer PostgreSQL")


## 5. Conclusion

### Pipeline réalisé

```
Nominatim API          → Coordonnées GPS des 35 villes françaises
        ↓
OpenWeatherMap API     → Prévisions météo 7 jours / ville
        ↓
Score météo composite  → Classement des destinations (temp + pluie + humidité)
        ↓
Booking.com scraping   → Hôtels avec notes utilisateurs (BeautifulSoup)
        ↓
Géocodage Nominatim    → Coordonnées précises par hôtel (avec repli + jitter)
        ↓
AWS S3 (Data Lake)     → Stockage raw + processed (CSV)
        ↓
AWS RDS (PostgreSQL)   → Data Warehouse requêtable par l'équipe analytics
        ↓
Plotly Maps            → 2 cartes interactives livrées à l'équipe marketing
```

### Résultats

| Indicateur | Valeur |
|---|---|
| Villes analysées | 35 |
| Jours de prévisions | 7 |
| Hôtels scrapés | 25 |
| Top destination | Collioure (score : 85.9) |
| 2ème destination | Nîmes (score : 84.4) |
| 3ème destination | Bayonne (score : 83.0) |

### Recommandation Kayak

Le **Sud de la France** (Collioure, Nîmes, Avignon, Aix-en-Provence) offre les meilleures conditions météo. Les équipes marketing peuvent cibler ces destinations pour leurs campagnes estivales.